# 00 — Setup, API Key Testing & Environment Check
**MSBA 305 | Maritime Shipping Intelligence Pipeline**

Run this notebook FIRST before anything else.
It verifies that all 3 API keys work and all libraries are installed.

---
## Your 3 Data Sources

| # | Source | Type | Format | API Key needed |
|---|--------|------|--------|----------------|
| 1 | UN Comtrade | Trade flows (historical) | CSV (already downloaded) | No |
| 2 | Nasdaq Data Link | Baltic Dry Index (daily) | JSON API | Yes — free |
| 3 | OpenWeatherMap | Weather at top ports (real-time) | JSON API | Yes — free |

---
## How to get your API keys (takes 5 minutes each)

**Nasdaq Data Link (BDI):**
1. Go to https://data.nasdaq.com/sign-up
2. Register free account
3. Go to Profile → API Key
4. Copy the key below

**OpenWeatherMap (Port weather):**
1. Go to https://openweathermap.org/api
2. Click 'Get API key' → Register free
3. Go to your account → API Keys
4. Copy the key below (free tier: 1000 calls/day)

**MarineTraffic (Optional bonus - vessel positions):**
1. Go to https://www.marinetraffic.com/en/ais-api
2. Register → free tier gives limited calls
3. Use if available — pipeline works without it

## Step 1 — Mount Google Drive & Set Paths

In [ ]:
import os, sys

# ── Environment detection: works in Colab and local/GitHub clone ──────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/repo'
    print('Running in Google Colab')
except ImportError:
    BASE = os.path.abspath(os.path.join(os.getcwd(), '..'))
    print(f'Running locally — BASE: {BASE}')

# create all folders if they don't exist
for folder in ['data/raw', 'data/clean', 'notebooks', 'scripts', 'SQL', 'dashboard']:
    os.makedirs(os.path.join(BASE, folder), exist_ok=True)

print('Folders ready')
print(f'Base path: {BASE}')


## Step 2 — Install & Import All Libraries

In [ ]:
!pip install nasdaqdatalink pyarrow google-cloud-bigquery pandas-gbq pydantic --quiet
print('All packages installed')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import nasdaqdatalink
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print('All imports successful')
print(f'Pandas version: {pd.__version__}')
print(f'Run date: {datetime.now().strftime("%Y-%m-%d %H:%M")}')

## Step 3 — Enter Your API Keys
**IMPORTANT:** Never commit API keys to GitHub. Store them here only (Colab runtime is private).

In [ ]:
# ── PASTE YOUR API KEYS HERE ─────────────────────────────────
NASDAQ_API_KEY  = 'YOUR_NASDAQ_KEY_HERE'        # from data.nasdaq.com
WEATHER_API_KEY = 'YOUR_OPENWEATHER_KEY_HERE'   # from openweathermap.org
MARINE_API_KEY  = 'YOUR_MARINETRAFFIC_KEY_HERE' # optional

# BigQuery project (set after creating GCP project)
BQ_PROJECT = 'your-gcp-project-id'
BQ_DATASET = 'shipping_data'
# ─────────────────────────────────────────────────────────────

print('Keys stored in memory (not saved to disk)')

## Step 4 — Test Source 1: UN Comtrade (CSV already downloaded)

In [ ]:
# ── Find the Comtrade CSV in your data/raw folder ──
import glob

raw_dir = f'{BASE}/data/raw'
csv_files = glob.glob(f'{raw_dir}/*.csv')
print(f'CSV files found in data/raw:')
for f in csv_files:
    size_mb = os.path.getsize(f) / 1e6
    print(f'  {os.path.basename(f)} ({size_mb:.1f} MB)')

# Try to load the Comtrade file
comtrade_files = [f for f in csv_files if 'Trade' in f or 'comtrade' in f.lower()]
if comtrade_files:
    sample = pd.read_csv(comtrade_files[0], nrows=3)
    print(f'\nComtrade file preview ({comtrade_files[0]}):')
    print(f'Columns: {sample.columns.tolist()}')
    print(f'Shape sample: {sample.shape}')
    print('SOURCE 1 (UN Comtrade): OK')
else:
    print('WARNING: No Comtrade CSV found. Upload TradeData file to data/raw/')

## Step 5 — Test Source 2: Nasdaq Data Link (Baltic Dry Index)

In [ ]:
nasdaqdatalink.ApiConfig.api_key = NASDAQ_API_KEY

try:
    bdi_test = nasdaqdatalink.get('CHRIS/ICE_BDR1', rows=5)
    print('SOURCE 2 (Nasdaq BDI): OK')
    print(f'Latest BDI date: {bdi_test.index[-1].date()}')
    print(f'Latest BDI value: {bdi_test["Settle"].iloc[-1]:.0f}')
    print(bdi_test.head(3))
except Exception as e:
    print(f'SOURCE 2 FAILED: {e}')
    print('Check your NASDAQ_API_KEY above')

## Step 6 — Test Source 3: OpenWeatherMap (Weather at Major Ports)

In [ ]:
# Test with Shanghai port coordinates
url = f'https://api.openweathermap.org/data/2.5/weather?lat=31.2304&lon=121.4737&appid={WEATHER_API_KEY}&units=metric'

try:
    resp = requests.get(url, timeout=10)
    if resp.status_code == 200:
        data = resp.json()
        print('SOURCE 3 (OpenWeatherMap): OK')
        print(f'Port: Shanghai | Temp: {data["main"]["temp"]}°C | Wind: {data["wind"]["speed"]} m/s')
        print(f'Weather: {data["weather"][0]["description"]}')
    else:
        print(f'SOURCE 3 FAILED: HTTP {resp.status_code}')
        print(resp.json())
except Exception as e:
    print(f'SOURCE 3 FAILED: {e}')
    print('Check your WEATHER_API_KEY above')

## Step 7 — Summary & Next Steps

In [ ]:
print('=' * 55)
print('SETUP COMPLETE — NEXT STEPS')
print('=' * 55)
print()
print('Run notebooks in this order:')
print('  01_clean_comtrade.ipynb    ← Source 1 (CSV)')
print('  02_ingest_clean_bdi.ipynb  ← Source 2 (Nasdaq API)')
print('  03_ingest_clean_weather.ipynb ← Source 3 (Weather API)')
print('  04_combine_EDA.ipynb       ← Merge + all charts')
print('  05_bigquery_upload.ipynb   ← Upload to BigQuery')
print()
print('GitHub Actions (automatic daily run):')
print('  .github/workflows/daily_update.yml')
print('  → Fetches BDI + Weather daily at 20:00 UTC')